# SkillPatch — Jupyter Test Notebook
Tests every layer of the pipeline in isolation, then wires them together.

**Upload checklist (same folder as this notebook):**
```
tinyvla_debugger/          ← the whole package folder
skillpatch_test.ipynb      ← this file
models/                    ← optional: put your .gguf file here
```

**Layer order tested here:**
1. Compiler (mock → llama_cpp)
2. Classifier
3. Patch Library
4. Audio Feedback (ElevenLabs)
5. Simulated camera frame
6. Simulated audio input → NL command
7. VLM API (vlm_api — Ollama/Moondream2, graceful degradation if offline)
8. Full orchestration — all mocks, happy path
9. Full orchestration — failure + auto-patch recovery
10. Full orchestration — unrecoverable abort
11. Full orchestration — real llama_cpp compiler end-to-end

## Cell 1 — Install dependencies

In [ ]:
# Run once. Re-run if you get ImportError below.
import subprocess, sys

pkgs = [
    "opencv-python-headless",
    "numpy",
    "elevenlabs",
]
for pkg in pkgs:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--break-system-packages", "-q"],
        check=True
    )
    print(f"  ✓ {pkg}")

# ROCm llama-cpp-python (run once on the node, only needed for Cell 4 / Cell 14):
# import os
# env = os.environ.copy()
# env["CMAKE_ARGS"] = "-DGGML_HIPBLAS=on"
# subprocess.run(
#     [sys.executable, "-m", "pip", "install", "llama-cpp-python",
#      "--force-reinstall", "--no-cache-dir", "--break-system-packages"],
#     env=env, check=True
# )

# Audio playback on the Linux node:
# subprocess.run(["sudo", "apt-get", "install", "-y", "mpg123", "-q"], check=True)

print("\nAll packages ready.")

## Cell 2 — Path setup & imports

In [ ]:
import sys, os, json, asyncio, logging
import numpy as np

# tinyvla_debugger folder must be in the same directory as this notebook.
# If it's one level up, change '.' to '..' below.
sys.path.insert(0, '.')

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s | %(name)s | %(message)s'
)

# --- Confirm package is importable ---
from tinyvla_debugger.compiler      import SkillCompiler
from tinyvla_debugger.classifier    import FailureClassifier
from tinyvla_debugger.patch_library import PatchLibrary
from tinyvla_debugger.audio_feedback import AudioFeedback
from tinyvla_debugger.orchestrator  import Orchestrator, SkillAbortError, MockRobotAPI, MockVLMAPI
from tinyvla_debugger               import trace_logger
from tinyvla_debugger               import vlm_api

print('✓ All imports OK')

## Cell 3 — Layer 1: Compiler (mock backend — no model needed)

In [ ]:
compiler = SkillCompiler(backend='mock')
print(f'Backend: {compiler.backend}\n')

test_commands = [
    'Put the canned goods on the middle shelf',
    'Refill slot 2 only',
    'Stock slots 1 and 2',
]

for cmd in test_commands:
    skill = compiler.compile(cmd)
    print(f'  "{cmd}"')
    print(f'  → skill_name : {skill["skill_name"]}')
    print(f'  → steps      : {len(skill["steps"])}')
    print(f'  → actions    : {[s["action"] for s in skill["steps"]]}')
    print()

# Also test SKILL_REGISTRY direct lookup (used by run_from_webcam)
from tinyvla_debugger.compiler import SKILL_REGISTRY
for name in SKILL_REGISTRY:
    s = compiler.compile(name)   # registry keys route without hitting LLM
    print(f'  registry[{name}] → {len(s["steps"])} steps')

print('\n✓ Mock compiler OK')

## Cell 4 — Layer 1: Compiler (llama_cpp — real Phi-3 inference)

In [ ]:
# Set this to wherever your .gguf file lives on the node.
GGUF_PATH = './models/Phi-3-mini-4k-instruct-q4.gguf'

if not os.path.exists(GGUF_PATH):
    print(f'⚠ Model not found at {GGUF_PATH}')
    print('  Skipping llama_cpp test. Set GGUF_PATH above to your model file.')
    print('  Download: huggingface-cli download microsoft/Phi-3-mini-4k-instruct-gguf '
          'Phi-3-mini-4k-instruct-q4.gguf --local-dir ./models/')
else:
    os.environ['PHI3_GGUF_PATH'] = GGUF_PATH
    llm_compiler = SkillCompiler(backend='llama_cpp')
    print(f'Backend: {llm_compiler.backend}')

    raw_skill = llm_compiler.compile('Stock the top shelf with the juice boxes')
    print(json.dumps(raw_skill, indent=2))

    # _extract_first_json_object strips trailing "Note: ..." text from Phi-3
    # _parse_and_validate also auto-unwraps if model wraps in a parent key
    print('\n✓ llama_cpp compiler OK')

## Cell 5 — Layer 4a: Failure Classifier

In [ ]:
clf = FailureClassifier()

# Each tuple: (action, verification_query, vlm_result, retry, expected_failure_type)
# Queries must match the pattern lists in classifier.py to get the right label.
test_cases = [
    ('pick_from_box',  'Is an object held securely in the gripper?',          False, False, 'GRASP_FAIL'),
    ('place_slot_1',   'Is there an item standing upright in shelf slot 1?',  False, False, 'PLACEMENT_MISS'),
    ('place_slot_1',   'Is there an item standing upright in shelf slot 1?',  False, True,  'PLACEMENT_COLLISION'),
    ('place_slot_2',   'Is this shelf slot currently empty?',                  True,  False, 'DROP_DURING_TRANSIT'),
    ('scan_scene',     'Is there an object visible in the pick zone?',         False, False, 'OBJECT_NOT_FOUND'),
]

print(f'{"Action":<20} {"Expected":<22} {"Got":<22} {"OK?"}')
print('-' * 75)
all_pass = True
for action, query, result, retry, expected in test_cases:
    r = clf.classify(action, query, result, retry=retry)
    ok = '✓' if r.failure_type == expected else '✗'
    if r.failure_type != expected:
        all_pass = False
        print(f'{action:<20} {expected:<22} {r.failure_type:<22} {ok}  ← MISMATCH')
    else:
        print(f'{action:<20} {expected:<22} {r.failure_type:<22} {ok}')

print(f'\n{"All tests passed!" if all_pass else "FAILURES DETECTED"}')

## Cell 6 — Layer 4b: Patch Library

In [ ]:
import tempfile, pathlib

# Use a temp file so tests don't pollute your real patches.json
tmp_patches = pathlib.Path(tempfile.mktemp(suffix='.json'))
lib = PatchLibrary(patches_file=str(tmp_patches))

base_params = {'z_offset_mm': 0.0, 'speed_scale': 1.0,
               'approach_angle_deg': 0.0, 'gripper_close_force': 0.6, 'retry_count': 2}

# --- store + retrieve a GRASP_FAIL patch ---
lib.store_patch('stock_middle_shelf', 'GRASP_FAIL', {'z_offset_mm': 5.0})
patch = lib.get_patch('stock_middle_shelf', 'GRASP_FAIL')
patched_params = lib.apply_to_params(base_params, patch)

print('Stored patch  :', patch)
print('Base params   :', base_params)
print('Patched params:', patched_params)
assert patched_params['z_offset_mm'] == 5.0, 'z_offset_mm delta not applied!'
print('✓ delta patch OK')

# --- absolute patch (speed_scale uses replace, not add) ---
lib.store_patch('stock_middle_shelf', 'PLACEMENT_COLLISION', {'speed_scale': 0.7})
patch2 = lib.get_patch('stock_middle_shelf', 'PLACEMENT_COLLISION')
patched2 = lib.apply_to_params(base_params, patch2)
assert patched2['speed_scale'] == 0.7, f'Expected 0.7, got {patched2["speed_scale"]}'
print('✓ absolute patch OK (speed_scale replaced, not added)')

# --- get_preexisting_patches merges all patches for a skill ---
pre = lib.get_preexisting_patches('stock_middle_shelf')
print('Pre-existing  :', pre)
assert 'z_offset_mm' in pre and 'speed_scale' in pre
print('✓ get_preexisting_patches OK')

tmp_patches.unlink(missing_ok=True)
print('\n✓ PatchLibrary OK')

## Cell 7 — Audio Feedback (ElevenLabs)

In [ ]:
# ── 7A: Mock mode (no API key needed) ──────────────────────────────────────
print('=== Mock mode (prints instead of speaking) ===')
audio_mock = AudioFeedback()   # silently mocks if ELEVENLABS_API_KEY not set
audio_mock.speak_success('stock_middle_shelf', steps_executed=4, steps_patched=1)
audio_mock.speak_patch_success('GRASP_FAIL', step_id=1)
audio_mock.speak_error('GRASP_FAIL', step_id=1, skill_name='stock_middle_shelf')

# ── 7B: Real ElevenLabs (needs API key + mpg123 on node) ───────────────────
# Set your key in the environment (recommended — keeps it out of the notebook):
#   export ELEVENLABS_API_KEY='sk_...'
# Or set it here for a quick test:
# os.environ['ELEVENLABS_API_KEY'] = 'sk_...'
#
# audio_real = AudioFeedback()
# audio_real.speak_success('stock_middle_shelf', steps_executed=4, steps_patched=1)
# audio_real.speak_error('GRASP_FAIL', step_id=1, skill_name='stock_middle_shelf')
#
# To use a different voice, find voice_id at:
#   GET https://api.elevenlabs.io/v1/voices  (Authorization: xi-api-key YOUR_KEY)
# audio_real = AudioFeedback(voice_id='YOUR_VOICE_ID_HERE')

print('\n✓ AudioFeedback OK')

## Cell 8 — Simulated camera frame (stands in for real webcam)

In [ ]:
import cv2

def get_simulated_frame(scene='shelf', width=640, height=480) -> np.ndarray:
    """BGR uint8 synthetic frame. Swap for get_real_frame() when webcam is live."""
    scenes = {
        'shelf':     (60,  120, 60),
        'gripper':   (80,  80,  160),
        'empty_box': (160, 160, 160),
    }
    color = scenes.get(scene, (100, 100, 100))
    frame = np.full((height, width, 3), color, dtype=np.uint8)
    cv2.putText(frame, f'SIM: {scene}', (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    return frame

def get_real_frame(device_index=0) -> np.ndarray:
    """Capture one frame from webcam. Use when USB cam is connected."""
    cap = cv2.VideoCapture(device_index)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f'Camera index {device_index} not available.')
    return frame

# Test sim frame
frame = get_simulated_frame(scene='shelf')
print(f'Frame shape : {frame.shape}  dtype: {frame.dtype}')
print(f'Frame range : [{frame.min()}, {frame.max()}]')

# TODO: HARDWARE — swap to get_real_frame() when webcam is connected:
# frame = get_real_frame(device_index=0)

print('\n✓ Camera frame OK')

## Cell 9 — Simulated audio input → NL command

In [ ]:
import random

# ── 9A: Simulation (pick a canned command) ─────────────────────────────────
DEMO_COMMANDS = [
    'Put the canned goods on the middle shelf',
    'Refill slot 2 only',
    'Stock slots 1 and 2',
    'Move the juice boxes to slot 3',
]

def get_simulated_audio_command() -> str:
    return random.choice(DEMO_COMMANDS)

# ── 9B: Real mic → Whisper transcription ────────────────────────────────────
# Requires: pip install sounddevice scipy openai-whisper --break-system-packages
#
# def get_real_audio_command(record_seconds=4, sample_rate=16000) -> str:
#     import sounddevice as sd
#     from scipy.io.wavfile import write
#     import whisper, tempfile
#     print(f'Recording {record_seconds}s... speak now!')
#     audio = sd.rec(int(record_seconds * sample_rate),
#                    samplerate=sample_rate, channels=1, dtype='int16')
#     sd.wait()
#     with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
#         write(f.name, sample_rate, audio)
#         result = whisper.load_model('tiny').transcribe(f.name)
#         os.unlink(f.name)
#     return result['text'].strip()

nl_command = get_simulated_audio_command()
# nl_command = get_real_audio_command()   # TODO: HARDWARE — uncomment when mic available

print(f'NL command: "{nl_command}"')
print('\n✓ Audio input OK')

## Cell 10 — VLM API (Ollama + Moondream2)
vlm_api.py is **fully implemented** by Sasha — it uses Ollama at `localhost:11434` with Moondream2.
This cell tests the graceful degradation path: if Ollama is offline, `verify()` returns `(False, 0.0)` and `locate_object()` returns all-False hints. The assertions pass regardless of whether Ollama is running.

In [ ]:
test_frame = get_simulated_frame('shelf')

# --- verify() ---
print('=== vlm_api.verify() — shortcut key test ===')
for query_key in ['grasp_check', 'slot_empty', 'slot_filled', 'object_visible']:
    result, latency = vlm_api.verify(test_frame, query_key)
    print(f'  {query_key:<25} → result={str(result):<5}  latency={latency:.0f}ms')
    assert isinstance(result, bool),  f'verify({query_key}) must return bool'
    assert isinstance(latency, float), f'verify({query_key}) must return float'

print('\n=== vlm_api.verify() — free-text passthrough ===')
result, latency = vlm_api.verify(test_frame, 'Is an object held securely in the gripper?')
print(f'  free-text → result={result}  latency={latency:.0f}ms')
assert isinstance(result, bool)

# --- locate_object() ---
print('\n=== vlm_api.locate_object() ===')
hints = vlm_api.locate_object(test_frame)
print(f'  hints = {hints}')
assert 'visible'    in hints, 'locate_object() must return "visible" key'
assert 'left'       in hints, 'locate_object() must return "left" key'
assert 'high'       in hints, 'locate_object() must return "high" key'
assert 'latency_ms' in hints, 'locate_object() must return "latency_ms" key'
print('✓ locate_object() keys OK')

# --- QUERIES dict ---
print('\n=== vlm_api.QUERIES keys ===')
for key, question in vlm_api.QUERIES.items():
    print(f'  {key:<25} → "{question[:65]}"')

print('\n✓ vlm_api module OK (graceful if Ollama offline)')

## Cell 11 — Full orchestration: all mocks, happy path
`_capture_frame` is monkeypatched to return a simulated frame so no USB webcam is needed.

In [ ]:
import pathlib, tempfile

tmp_dir    = pathlib.Path(tempfile.mkdtemp())
trace_file = tmp_dir / 'trace.jsonl'
patch_file = tmp_dir / 'patches.json'

# MockVLMAPI(fail_step_ids=set()) → always returns True (no failures)
robot = MockRobotAPI(failure_on_step=-1)
vlm   = MockVLMAPI(fail_step_ids=set())
audio = AudioFeedback(enabled=True)

orc = Orchestrator(
    robot=robot,
    vlm=vlm,
    audio=audio,
    compiler_backend='mock',
    patches_file=str(patch_file),
    trace_file=str(trace_file),
)
# Monkeypatch so orchestrator doesn't try to open a real webcam
orc._capture_frame = lambda: get_simulated_frame('shelf')

result = await orc.run_skill(nl_command)

print(f'\n=== Result ===')
print(f'  Outcome        : {result.outcome}')
print(f'  Skill name     : {result.skill_name}')
print(f'  Steps executed : {result.steps_executed}')
print(f'  Steps patched  : {result.steps_patched}')
print(f'  Total GPU ms   : {result.total_gpu_ms:.1f}')
assert result.outcome == 'success', f'Expected success, got {result.outcome}'
assert result.steps_patched == 0,   f'Happy path should have 0 patches'
print('\n✓ Happy path OK')

## Cell 12 — Full orchestration: failure → auto-patch → recovery
`PlacementFailVLM` passes all queries EXCEPT the very first placement-check query
(`"upright in shelf slot"`), which maps to `PLACEMENT_MISS` in the classifier and has a
default patch (`approach_angle_deg +10`). The retry then passes, `patched_success` is
logged, and `steps_patched` increments.

Expected result: `outcome=success`, `steps_patched >= 1`.

In [ ]:
import pathlib, tempfile

tmp_dir2    = pathlib.Path(tempfile.mkdtemp())
trace_file2 = tmp_dir2 / 'trace.jsonl'
patch_file2 = tmp_dir2 / 'patches.json'


class PlacementFailVLM:
    """
    Fails on the FIRST placement-check query, passes everything else.

    Placement queries contain "upright in shelf slot" — generated by the mock compiler
    for place_slot_* steps.  The classifier maps this to PLACEMENT_MISS, which has
    the default patch: approach_angle_deg +10.  patch_applied is set to True, and on
    the retry the step passes → patched_success is logged, steps_patched increments.

    Flow for "Put the canned goods on the middle shelf":
      scan_shelf   → "Is the middle shelf stocked?"              → True  (PASS)
      place_slot_1 → "Is there an item upright in shelf slot 1?" → False (PLACEMENT_MISS)
          patch: approach_angle_deg +10 applied
          retry  → True  → patched_success, steps_patched = 1
      place_slot_2, place_slot_3, check_box_empty                → True  (PASS)
    """
    def __init__(self, latency_ms: float = 95.0):
        self._placement_calls = 0
        self.latency_ms = latency_ms

    def verify(self, frame, query: str):
        if "upright in shelf slot" in query.lower() or "standing upright" in query.lower():
            self._placement_calls += 1
            if self._placement_calls == 1:
                return False, self.latency_ms   # first placement query fails → PLACEMENT_MISS
        return True, self.latency_ms            # everything else passes


robot2 = MockRobotAPI()
vlm2   = PlacementFailVLM()
audio2 = AudioFeedback(enabled=True)

orc2 = Orchestrator(
    robot=robot2,
    vlm=vlm2,
    audio=audio2,
    compiler_backend='mock',
    patches_file=str(patch_file2),
    trace_file=str(trace_file2),
)
orc2._capture_frame = lambda: get_simulated_frame('shelf')

result2 = await orc2.run_skill('Put the canned goods on the middle shelf')

print(f'\n=== Result ===')
print(f'  Outcome        : {result2.outcome}')
print(f'  Steps patched  : {result2.steps_patched}   ← should be ≥ 1')
assert result2.outcome == 'success',     f'Expected success, got {result2.outcome}'
assert result2.steps_patched >= 1,       f'Expected steps_patched >= 1, got {result2.steps_patched}'
print('\n✓ Patch recovery path OK')

print('\n=== Trace (newest first) ===')
for evt in trace_logger.read_trace(trace_file=trace_file2):
    step  = evt.get('step_id', '?')
    res   = str(evt.get('result', '?'))
    patch = evt.get('patch_applied')
    gpu   = evt.get('gpu_latency_ms')
    print(f'  step={step}  result={res:<22}  patch={patch}  gpu_ms={gpu}')

## Cell 13 — Full orchestration: unrecoverable abort
`AlwaysFailVLM` returns `False` on every call — both attempts at every step fail,
so `SkillAbortError` is raised and `speak_error()` fires.

In [ ]:
import pathlib, tempfile

tmp_dir3    = pathlib.Path(tempfile.mkdtemp())
trace_file3 = tmp_dir3 / 'trace.jsonl'
patch_file3 = tmp_dir3 / 'patches.json'


class AlwaysFailVLM:
    """Always returns False — simulates a non-recoverable hardware failure."""
    def verify(self, frame, query: str):
        return False, 95.0


audio3 = AudioFeedback(enabled=True)
orc3 = Orchestrator(
    robot=MockRobotAPI(),
    vlm=AlwaysFailVLM(),
    audio=audio3,
    compiler_backend='mock',
    patches_file=str(patch_file3),
    trace_file=str(trace_file3),
)
orc3._capture_frame = lambda: get_simulated_frame('shelf')

try:
    await orc3.run_skill('Refill slot 2 only')
    print('ERROR: Should have raised SkillAbortError!')
except SkillAbortError as e:
    print(f'✓ SkillAbortError raised as expected')
    print(f'  failure_type : {e.failure_type}')
    print(f'  step_id      : {e.step_id}')
    print(f'  skill_name   : {e.skill_name}')
    print('  → speak_error() fired just before this raise')

## Cell 14 — Full orchestration: real llama_cpp compiler end-to-end
Requires the GGUF model from Cell 4. Skips gracefully if not found.

In [ ]:
import pathlib, tempfile

GGUF_PATH = './models/Phi-3-mini-4k-instruct-q4.gguf'

if not os.path.exists(GGUF_PATH):
    print(f'⚠ Skipping: model not found at {GGUF_PATH}')
else:
    os.environ['PHI3_GGUF_PATH'] = GGUF_PATH

    tmp_dir4    = pathlib.Path(tempfile.mkdtemp())
    trace_file4 = tmp_dir4 / 'trace.jsonl'

    command = get_simulated_audio_command()
    # command = get_real_audio_command()   # TODO: HARDWARE — real mic
    print(f'Command: "{command}"\n')

    orc4 = Orchestrator(
        robot=MockRobotAPI(),
        vlm=MockVLMAPI(fail_step_ids=set()),
        audio=AudioFeedback(enabled=True),
        compiler_backend='llama_cpp',
        trace_file=str(trace_file4),
    )
    orc4._capture_frame = lambda: get_simulated_frame('shelf')

    result4 = await orc4.run_skill(command)
    print(f'Outcome    : {result4.outcome}')
    print(f'Skill name : {result4.skill_name}')
    print(f'Steps      : {result4.steps_executed}')
    print(f'GPU ms     : {result4.total_gpu_ms:.1f}')

    print('\n=== Steps from Phi-3 ===')
    for evt in reversed(trace_logger.read_trace(trace_file=trace_file4)):
        if evt.get('step_id') is not None:
            print(f'  step={evt["step_id"]}  action={str(evt.get("action","")):<22}  result={evt.get("result")}')

## Cell 15 — TODO: HARDWARE checklist
Every item below is currently mocked. Replace one at a time as hardware comes online.

In [ ]:
todo = [
    # Layer                  Current state               What to do
    ('NL compiler',          'mock → llama_cpp',          'Set GGUF_PATH; rebuild llama-cpp-python with CMAKE_ARGS="-DGGML_HIPBLAS=on"'),
    ('VLM verifier',         'MockVLMAPI (sim)',           'Already implemented (Sasha). Start Ollama: ollama serve + ollama pull moondream'),
    ('Robot hardware',       'MockRobotAPI (sim)',         'Diya: implement robot_api.replay_skill() / apply_patch() for SO-100 arm'),
    ('Camera frame',         'get_simulated_frame()',      'Swap orc._capture_frame to get_real_frame(device_index=0) or remove monkeypatch'),
    ('Audio input',          'get_simulated_audio_command()', 'Swap to get_real_audio_command() with Whisper; pip install sounddevice scipy openai-whisper'),
    ('ElevenLabs TTS',       'mock (print-only)',          'Set ELEVENLABS_API_KEY env var + sudo apt-get install -y mpg123'),
    ('run_from_webcam()',     'not tested yet',            'Call await orc.run_from_webcam() once camera + Ollama are both live'),
]

print(f'{"Layer":<22} {"Current":<30} {"Action"}')
print('-' * 100)
for layer, current, action in todo:
    print(f'{layer:<22} {current:<30} {action}')